In [21]:
# =============================================================================
# Add Project Root To Python Path
# =============================================================================

import sys
from pathlib import Path


PROJECT_ROOT = Path.cwd()


if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent


if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(
        str(PROJECT_ROOT)
    )


print("Project Root:")
print(PROJECT_ROOT)

Project Root:
c:\Users\abuba\Downloads\ML-Projects\Day-5


In [22]:
# =============================================================================
# Imports
# =============================================================================

import pandas as pd
import numpy as np

import time
import logging


from pathlib import Path


from sklearn.model_selection import RandomizedSearchCV


from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
)


from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
)


from src.data.load_data import (
    load_featured_data,
)


from src.models.utils import (
    save_model,
    load_model,
    save_results,
)

In [23]:
# =============================================================================
# Project Paths
# =============================================================================

from pathlib import Path


PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent


MODELS_DIR = PROJECT_ROOT / "models"

TREE_MODELS_DIR = MODELS_DIR / "tree"

REPORTS_DIR = PROJECT_ROOT / "reports"


print("=" * 70)
print("Project Paths")
print("=" * 70)

print(f"Project Root : {PROJECT_ROOT}")
print(f"Models       : {TREE_MODELS_DIR}")
print(f"Reports      : {REPORTS_DIR}")

Project Paths
Project Root : c:\Users\abuba\Downloads\ML-Projects\Day-5
Models       : c:\Users\abuba\Downloads\ML-Projects\Day-5\models\tree
Reports      : c:\Users\abuba\Downloads\ML-Projects\Day-5\reports


In [24]:
## Configure Logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)


logger = logging.getLogger(__name__)


logger.info("Logging configured successfully")

2026-07-23 16:49:54,850 | INFO | Logging configured successfully


In [25]:
# =============================================================================
# Load Feature Engineered Dataset
# =============================================================================

featured_df = load_featured_data()


print("=" * 70)
print("Feature Engineered Dataset Loaded")
print("=" * 70)


print(f"Dataset Shape : {featured_df.shape}")

featured_df.head()

Feature Engineered Dataset Loaded
Dataset Shape : (18378485, 32)


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,congestion_surcharge,...,hour_sin,hour_cos,day_sin,day_cos,same_borough,same_location,is_airport_trip,distance_squared,log_trip_distance,distance_per_passenger
0,2,2023-01-01 00:32:10,2023-01-01 00:40:36,1,0.97,1.0,N,161,141,2.5,...,0.0,1.0,-0.781832,0.62349,1,0,0,0.9409,0.678034,0.97
1,2,2023-01-01 00:55:08,2023-01-01 01:01:27,1,1.10,1.0,N,43,237,2.5,...,0.0,1.0,-0.781832,0.62349,1,0,0,1.2100,0.741937,1.10
2,2,2023-01-01 00:25:04,2023-01-01 00:37:49,1,2.51,1.0,N,48,238,2.5,...,0.0,1.0,-0.781832,0.62349,1,0,0,6.3001,1.255616,2.51
3,2,2023-01-01 00:10:29,2023-01-01 00:21:19,1,1.43,1.0,N,107,79,2.5,...,0.0,1.0,-0.781832,0.62349,1,0,0,2.0449,0.887891,1.43
4,2,2023-01-01 00:50:34,2023-01-01 01:02:52,1,1.84,1.0,N,161,137,2.5,...,0.0,1.0,-0.781832,0.62349,1,0,0,3.3856,1.043804,1.84


Dataset ValidationPrepare Features and Target

In [26]:

TARGET = "log_trip_duration"


DROP_COLUMNS = [
    "log_trip_duration",
    "trip_duration_minutes",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
]


X = featured_df.drop(
    columns=DROP_COLUMNS
)


y = featured_df[TARGET]


print("=" * 70)
print("Dataset Prepared")
print("=" * 70)


print(f"Feature Matrix Shape : {X.shape}")
print(f"Target Shape         : {y.shape}")
print(f"Number of Features   : {X.shape[1]}")


X.head()

Dataset Prepared
Feature Matrix Shape : (18378485, 28)
Target Shape         : (18378485,)
Number of Features   : 28


,VendorID,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,congestion_surcharge,airport_fee,pickup_year,...,hour_sin,hour_cos,day_sin,day_cos,same_borough,same_location,is_airport_trip,distance_squared,log_trip_distance,distance_per_passenger
0,2,1,0.97,1.0,N,161,141,2.5,0.0,2023,...,0.0,1.0,-0.781832,0.62349,1,0,0,0.9409,0.678034,0.97
1,2,1,1.10,1.0,N,43,237,2.5,0.0,2023,...,0.0,1.0,-0.781832,0.62349,1,0,0,1.2100,0.741937,1.10
2,2,1,2.51,1.0,N,48,238,2.5,0.0,2023,...,0.0,1.0,-0.781832,0.62349,1,0,0,6.3001,1.255616,2.51
3,2,1,1.43,1.0,N,107,79,2.5,0.0,2023,...,0.0,1.0,-0.781832,0.62349,1,0,0,2.0449,0.887891,1.43
4,2,1,1.84,1.0,N,161,137,2.5,0.0,2023,...,0.0,1.0,-0.781832,0.62349,1,0,0,3.3856,1.043804,1.84


In [27]:
print(X.dtypes)

VendorID                    int64
passenger_count              int8
trip_distance             float32
RatecodeID                float64
store_and_fwd_flag            str
PULocationID                int64
DOLocationID                int64
congestion_surcharge      float64
airport_fee               float64
pickup_year                 int16
pickup_month                 int8
pickup_day                   int8
pickup_hour                  int8
pickup_dayofweek             int8
pickup_week                 int16
pickup_weekend               int8
pickup_quarter               int8
rush_hour                    int8
hour_sin                  float32
hour_cos                  float32
day_sin                   float32
day_cos                   float32
same_borough                 int8
same_location                int8
is_airport_trip              int8
distance_squared          float32
log_trip_distance         float32
distance_per_passenger    float32
dtype: object


In [28]:
# =============================================================================
# Train Test Split
# =============================================================================

from sklearn.model_selection import train_test_split


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


print("=" * 70)
print("Train Test Split Completed")
print("=" * 70)


print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

Train Test Split Completed
X_train: (14702788, 28)
X_test : (3675697, 28)
y_train: (14702788,)
y_test : (3675697,)


In [29]:
# =============================================================================
# Identify Feature Types
# =============================================================================

numeric_features = X_train.select_dtypes(
    include=[
        "int8",
        "int16",
        "int32",
        "int64",
        "float32",
        "float64"
    ]
).columns.tolist()


categorical_features = X_train.select_dtypes(
    include=[
        "object",
        "string"
    ]
).columns.tolist()


print("=" * 70)
print("Feature Types")
print("=" * 70)


print("Numeric Features:")
print(numeric_features)


print("\nCategorical Features:")
print(categorical_features)

Feature Types
Numeric Features:
['VendorID', 'passenger_count', 'trip_distance', 'RatecodeID', 'PULocationID', 'DOLocationID', 'congestion_surcharge', 'airport_fee', 'pickup_year', 'pickup_month', 'pickup_day', 'pickup_hour', 'pickup_dayofweek', 'pickup_week', 'pickup_weekend', 'pickup_quarter', 'rush_hour', 'hour_sin', 'hour_cos', 'day_sin', 'day_cos', 'same_borough', 'same_location', 'is_airport_trip', 'distance_squared', 'log_trip_distance', 'distance_per_passenger']

Categorical Features:
['store_and_fwd_flag']


In [30]:
# =============================================================================
# Preprocessing Pipeline
# =============================================================================

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder


preprocessor = ColumnTransformer(
    transformers=[
        
        (
            "numeric",
            "passthrough",
            numeric_features
        ),

        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        )
    ]
)


print("Preprocessor created successfully")

Preprocessor created successfully


In [31]:
# =============================================================================
# Transform Train and Test Data
# =============================================================================

X_train_processed = preprocessor.fit_transform(
    X_train
)


X_test_processed = preprocessor.transform(
    X_test
)


print("=" * 70)
print("Preprocessing Completed")
print("=" * 70)


print(
    "X_train_processed:",
    X_train_processed.shape
)


print(
    "X_test_processed:",
    X_test_processed.shape
)

Preprocessing Completed
X_train_processed: (14702788, 29)
X_test_processed: (3675697, 29)


In [32]:
# =============================================================================
# Save Preprocessor
# =============================================================================

save_model(
    preprocessor,
    "preprocessor",
    MODELS_DIR
)

2026-07-23 16:50:49,242 | INFO | Model saved to: c:\Users\abuba\Downloads\ML-Projects\Day-5\models\preprocessor.joblib


WindowsPath('c:/Users/abuba/Downloads/ML-Projects/Day-5/models/preprocessor.joblib')

In [34]:
# =============================================================================
# Prepare Training Sample For Hyperparameter Tuning
# =============================================================================

TUNING_SAMPLE_SIZE = 500000


X_train_tune = X_train_processed[:TUNING_SAMPLE_SIZE]

y_train_tune = y_train.iloc[:TUNING_SAMPLE_SIZE]


print("=" * 70)
print("Tuning Dataset Prepared")
print("=" * 70)


print(
    "X_train_tune Shape:",
    X_train_tune.shape
)


print(
    "y_train_tune Shape:",
    y_train_tune.shape
)

Tuning Dataset Prepared
X_train_tune Shape: (500000, 29)
y_train_tune Shape: (500000,)


In [35]:
# =============================================================================
# Tuning Results Storage
# =============================================================================

tuning_results = []


best_models = {}


print("Tuning storage initialized")

Tuning storage initialized


In [15]:
# =============================================================================
# Random Forest Hyperparameter Search Space
# =============================================================================

rf_param_grid = {

    "n_estimators": [
        50,
        100,
        150
    ],

    "max_depth": [
        10,
        20,
        30
    ],

    "min_samples_split": [
        2,
        5,
        10
    ],

    "min_samples_leaf": [
        1,
        2,
        4
    ]
}


print("=" * 70)
print("Random Forest Parameter Grid Created")
print("=" * 70)

print(rf_param_grid)

Random Forest Parameter Grid Created
{'n_estimators': [50, 100, 150], 'max_depth': [10, 20, 30], 'min_samples_split': [2, 5, 10], 'min_samples_leaf': [1, 2, 4]}


In [16]:
# =============================================================================
# Random Forest Base Model
# =============================================================================

rf_model = RandomForestRegressor(
    random_state=42,
    n_jobs=-1
)


print("Random Forest model created")

Random Forest model created


## Run RandomizedSearchCV

In [17]:
start_time = time.perf_counter()


rf_search = RandomizedSearchCV(
    estimator=rf_model,
    param_distributions=rf_param_grid,
    n_iter=10,
    cv=3,
    scoring="neg_root_mean_squared_error",
    random_state=42,
    n_jobs=-1,
    verbose=2
)


rf_search.fit(
    X_train_tune,
    y_train_tune
)


rf_time = time.perf_counter() - start_time


print("=" * 70)
print("Random Forest Tuning Completed")
print("=" * 70)

print(
    f"Training Time: {rf_time:.2f} seconds"
)

Fitting 3 folds for each of 10 candidates, totalling 30 fits
Random Forest Tuning Completed
Training Time: 792.61 seconds


In [20]:
# =============================================================================
# Extract Best Random Forest Model
# =============================================================================

best_random_forest = rf_search.best_estimator_


print("=" * 70)
print("Best Random Forest Model")
print("=" * 70)


print(best_random_forest)

Best Random Forest Model
RandomForestRegressor(max_depth=30, min_samples_leaf=2, min_samples_split=10,
                      n_jobs=-1, random_state=42)


In [21]:
save_model(
    best_random_forest,
    "tuned_random_forest",
    TREE_MODELS_DIR
)

2026-07-23 01:56:42,621 | INFO | Model saved to: c:\Users\abuba\Downloads\ML-Projects\Day-5\models\tree\tuned_random_forest.joblib


WindowsPath('c:/Users/abuba/Downloads/ML-Projects/Day-5/models/tree/tuned_random_forest.joblib')

In [19]:
print(rf_search)

RandomizedSearchCV(cv=3,
                   estimator=RandomForestRegressor(n_jobs=-1, random_state=42),
                   n_jobs=-1,
                   param_distributions={'max_depth': [10, 20, 30],
                                        'min_samples_leaf': [1, 2, 4],
                                        'min_samples_split': [2, 5, 10],
                                        'n_estimators': [50, 100, 150]},
                   random_state=42, scoring='neg_root_mean_squared_error',
                   verbose=2)


In [22]:
from sklearn.tree import DecisionTreeRegressor

In [23]:
# =============================================================================
# Decision Tree Hyperparameter Search Space
# =============================================================================

dt_param_grid = {

    "max_depth": [
        10,
        20,
        30,
        None
    ],

    "min_samples_split": [
        2,
        5,
        10
    ],

    "min_samples_leaf": [
        1,
        2,
        4
    ]
}


print("=" * 70)
print("Decision Tree Parameter Grid Created")
print("=" * 70)

print(dt_param_grid)

Decision Tree Parameter Grid Created
{'max_depth': [10, 20, 30, None], 'min_samples_split': [2, 5, 10], 'min_samples_leaf': [1, 2, 4]}


In [24]:
# =============================================================================
# Decision Tree Base Model
# =============================================================================

dt_model = DecisionTreeRegressor(
    random_state=42
)


print(dt_model)

DecisionTreeRegressor(random_state=42)


In [25]:
# =============================================================================
# Decision Tree Randomized Search
# =============================================================================

start_time = time.perf_counter()


dt_search = RandomizedSearchCV(
    estimator=dt_model,
    param_distributions=dt_param_grid,
    n_iter=5,
    cv=3,
    scoring="neg_root_mean_squared_error",
    random_state=42,
    n_jobs=-1,
    verbose=2
)


dt_search.fit(
    X_train_tune,
    y_train_tune
)


dt_time = time.perf_counter() - start_time


print("=" * 70)
print("Decision Tree Tuning Completed")
print("=" * 70)

print(
    f"Training Time: {dt_time:.2f} seconds"
)

Fitting 3 folds for each of 5 candidates, totalling 15 fits
Decision Tree Tuning Completed
Training Time: 18.85 seconds


In [26]:
# =============================================================================
# Best Decision Tree Parameters
# =============================================================================

print("=" * 70)
print("Best Decision Tree Parameters")
print("=" * 70)


print(dt_search.best_params_)

Best Decision Tree Parameters
{'min_samples_split': 5, 'min_samples_leaf': 4, 'max_depth': 20}


In [27]:
best_decision_tree = dt_search.best_estimator_


print(best_decision_tree)

DecisionTreeRegressor(max_depth=20, min_samples_leaf=4, min_samples_split=5,
                      random_state=42)


In [28]:
# =============================================================================
# Save Tuned Decision Tree
# =============================================================================

save_model(
    best_decision_tree,
    "tuned_decision_tree",
    TREE_MODELS_DIR
)


print("Tuned Decision Tree saved successfully")

2026-07-23 02:00:30,138 | INFO | Model saved to: c:\Users\abuba\Downloads\ML-Projects\Day-5\models\tree\tuned_decision_tree.joblib


Tuned Decision Tree saved successfully


In [36]:
from sklearn.ensemble import GradientBoostingRegressor

In [37]:
# =============================================================================
# Gradient Boosting Hyperparameter Search Space
# =============================================================================

gb_param_grid = {

    "n_estimators": [
        50,
        100
    ],

    "learning_rate": [
        0.05,
        0.1
    ],

    "max_depth": [
        3,
        5
    ],

    "min_samples_split": [
        2,
        5
    ],

    "min_samples_leaf": [
        1,
        2
    ]
}


print("=" * 70)
print("Gradient Boosting Parameter Grid Created")
print("=" * 70)

print(gb_param_grid)

Gradient Boosting Parameter Grid Created
{'n_estimators': [50, 100], 'learning_rate': [0.05, 0.1], 'max_depth': [3, 5], 'min_samples_split': [2, 5], 'min_samples_leaf': [1, 2]}


In [38]:
gb_model = GradientBoostingRegressor(
    random_state=42
)


print(gb_model)

GradientBoostingRegressor(random_state=42)


In [39]:
# =============================================================================
# Gradient Boosting Randomized Search
# =============================================================================

start_time = time.perf_counter()


gb_search = RandomizedSearchCV(
    estimator=gb_model,
    param_distributions=gb_param_grid,
    n_iter=5,
    cv=3,
    scoring="neg_root_mean_squared_error",
    random_state=42,
    n_jobs=-1,
    verbose=2
)


gb_search.fit(
    X_train_tune,
    y_train_tune
)


gb_time = time.perf_counter() - start_time


print("=" * 70)
print("Gradient Boosting Tuning Completed")
print("=" * 70)


print(
    f"Training Time: {gb_time:.2f} seconds"
)

Fitting 3 folds for each of 5 candidates, totalling 15 fits
Gradient Boosting Tuning Completed
Training Time: 704.59 seconds


In [40]:
# =============================================================================
# Best Gradient Boosting Model
# =============================================================================

best_gradient_boosting = gb_search.best_estimator_


print("="*70)
print("Best Parameters")
print("="*70)


print(
    gb_search.best_params_
)


print(best_gradient_boosting)

Best Parameters
{'n_estimators': 100, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_depth': 5, 'learning_rate': 0.1}
GradientBoostingRegressor(max_depth=5, min_samples_leaf=2, random_state=42)


In [41]:
# =============================================================================
# Training Performance Check
# =============================================================================

train_predictions = best_gradient_boosting.predict(
    X_train_tune
)


train_rmse = np.sqrt(
    mean_squared_error(
        y_train_tune,
        train_predictions
    )
)


train_r2 = r2_score(
    y_train_tune,
    train_predictions
)


print("="*70)
print("Training Performance")
print("="*70)

print(
    f"Train RMSE : {train_rmse:.4f}"
)

print(
    f"Train R2   : {train_r2:.4f}"
)

Training Performance
Train RMSE : 0.2644
Train R2   : 0.8453


In [42]:
# =============================================================================
# Save Tuned Gradient Boosting
# =============================================================================

save_model(
    best_gradient_boosting,
    "tuned_gradient_boosting",
    TREE_MODELS_DIR
)


print("Tuned Gradient Boosting saved successfully")

2026-07-23 17:05:39,254 | INFO | Model saved to: c:\Users\abuba\Downloads\ML-Projects\Day-5\models\tree\tuned_gradient_boosting.joblib


Tuned Gradient Boosting saved successfully


In [36]:
# =============================================================================
# Imports
# =============================================================================

from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV

In [37]:
# =============================================================================
# Ridge Hyperparameter Grid
# =============================================================================

ridge_param_grid = {
    "alpha": [
        0.001,
        0.01,
        0.1,
        1,
        10,
        100,
    ]
}

print("=" * 70)
print("Ridge Hyperparameter Grid Created")
print("=" * 70)

print(ridge_param_grid)

Ridge Hyperparameter Grid Created
{'alpha': [0.001, 0.01, 0.1, 1, 10, 100]}


In [38]:
# =============================================================================
# Ridge Regression Hyperparameter Tuning
# =============================================================================

start_time = time.perf_counter()

ridge_search = GridSearchCV(
    estimator=Ridge(random_state=42),
    param_grid=ridge_param_grid,
    scoring="neg_root_mean_squared_error",
    cv=3,
    n_jobs=-1,
)

ridge_search.fit(
    X_train_tune,
    y_train_tune,
)

ridge_time = time.perf_counter() - start_time

best_ridge = ridge_search.best_estimator_

print("=" * 70)
print("Ridge Regression Tuning Completed")
print("=" * 70)

print(f"Best Parameters : {ridge_search.best_params_}")
print(f"Best RMSE       : {-ridge_search.best_score_:.4f}")
print(f"Training Time   : {ridge_time:.2f} seconds")

Ridge Regression Tuning Completed
Best Parameters : {'alpha': 100}
Best RMSE       : 0.3189
Training Time   : 7.35 seconds


In [45]:
# =============================================================================
# Model Directories
# =============================================================================

LINEAR_MODELS_DIR = PROJECT_ROOT / "models" / "linear"

TREE_MODELS_DIR = PROJECT_ROOT / "models" / "tree"


LINEAR_MODELS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

TREE_MODELS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Linear Models :", LINEAR_MODELS_DIR)
print("Tree Models   :", TREE_MODELS_DIR)


Linear Models : c:\Users\abuba\Downloads\ML-Projects\Day-5\models\linear
Tree Models   : c:\Users\abuba\Downloads\ML-Projects\Day-5\models\tree


In [46]:
# =============================================================================
# Extract Best Ridge Model
# =============================================================================

best_ridge = ridge_search.best_estimator_


print("=" * 70)
print("Best Ridge Model")
print("=" * 70)

print(best_ridge)

print()

print("Best Parameters:")
print(ridge_search.best_params_)

Best Ridge Model
Ridge(alpha=100, random_state=42)

Best Parameters:
{'alpha': 100}


In [47]:
# =============================================================================
# Save Tuned Ridge Model
# =============================================================================

save_model(
    best_ridge,
    "tuned_ridge",
    LINEAR_MODELS_DIR,
)

print("Tuned Ridge model saved successfully.")

2026-07-23 02:38:23,219 | INFO | Model saved to: c:\Users\abuba\Downloads\ML-Projects\Day-5\models\linear\tuned_ridge.joblib


Tuned Ridge model saved successfully.


In [40]:
# =============================================================================
# Imports
# =============================================================================

from sklearn.linear_model import Lasso

In [41]:
# =============================================================================
# Lasso Hyperparameter Grid
# =============================================================================

lasso_param_grid = {
    "alpha": [
        0.0001,
        0.001,
        0.01,
        0.1,
        1,
    ]
}

print("=" * 70)
print("Lasso Hyperparameter Grid Created")
print("=" * 70)

print(lasso_param_grid)

Lasso Hyperparameter Grid Created
{'alpha': [0.0001, 0.001, 0.01, 0.1, 1]}


In [42]:
# =============================================================================
# Lasso Regression Hyperparameter Tuning
# =============================================================================

start_time = time.perf_counter()

lasso_search = GridSearchCV(
    estimator=Lasso(
        random_state=42,
        max_iter=10000,
    ),
    param_grid=lasso_param_grid,
    scoring="neg_root_mean_squared_error",
    cv=3,
    n_jobs=-1,
)

lasso_search.fit(
    X_train_tune,
    y_train_tune,
)

lasso_time = time.perf_counter() - start_time

best_lasso = lasso_search.best_estimator_

print("=" * 70)
print("Lasso Regression Tuning Completed")
print("=" * 70)

print(f"Best Parameters : {lasso_search.best_params_}")
print(f"Best RMSE       : {-lasso_search.best_score_:.4f}")
print(f"Training Time   : {lasso_time:.2f} seconds")

Lasso Regression Tuning Completed
Best Parameters : {'alpha': 0.0001}
Best RMSE       : 0.3190
Training Time   : 971.69 seconds


In [48]:
# =============================================================================
# Extract Best Lasso Model
# =============================================================================

best_lasso = lasso_search.best_estimator_


print("=" * 70)
print("Best Lasso Model")
print("=" * 70)

print(best_lasso)
print()
print("Best Parameters:")
print(lasso_search.best_params_)

Best Lasso Model
Lasso(alpha=0.0001, max_iter=10000, random_state=42)

Best Parameters:
{'alpha': 0.0001}


In [49]:
# =============================================================================
# Save Tuned Lasso Model
# =============================================================================

save_model(
    best_lasso,
    "tuned_lasso",
    LINEAR_MODELS_DIR,
)

print("Tuned Lasso model saved successfully.")

2026-07-23 02:38:32,612 | INFO | Model saved to: c:\Users\abuba\Downloads\ML-Projects\Day-5\models\linear\tuned_lasso.joblib


Tuned Lasso model saved successfully.


In [50]:
# =============================================================================
# Imports
# =============================================================================

from sklearn.linear_model import ElasticNet

In [53]:
# =============================================================================
# Elastic Net Hyperparameter Grid
# =============================================================================

elastic_param_grid = {
    "alpha": [
        0.001,
        0.01,
        0.1,
        1,
    ],

    "l1_ratio": [
        0.2,
        0.5,
        0.8,
    ],
}

print("=" * 70)
print("Elastic Net Hyperparameter Grid Created")
print("=" * 70)

print(elastic_param_grid)

Elastic Net Hyperparameter Grid Created
{'alpha': [0.001, 0.01, 0.1, 1], 'l1_ratio': [0.2, 0.5, 0.8]}


In [54]:
# =============================================================================
# Elastic Net Hyperparameter Tuning
# =============================================================================

start_time = time.perf_counter()


elastic_search = GridSearchCV(
    estimator=ElasticNet(
        random_state=42,
        max_iter=5000,
    ),
    param_grid=elastic_param_grid,
    scoring="neg_root_mean_squared_error",
    cv=3,
    n_jobs=-1,
    verbose=2,
)


elastic_search.fit(
    X_train_tune,
    y_train_tune,
)


elastic_time = time.perf_counter() - start_time


best_elastic_net = elastic_search.best_estimator_


print("=" * 70)
print("Elastic Net Regression Tuning Completed")
print("=" * 70)


print(
    f"Best Parameters : {elastic_search.best_params_}"
)

print(
    f"Best RMSE       : {-elastic_search.best_score_:.4f}"
)

print(
    f"Training Time   : {elastic_time:.2f} seconds"
)

Fitting 3 folds for each of 12 candidates, totalling 36 fits
Elastic Net Regression Tuning Completed
Best Parameters : {'alpha': 0.001, 'l1_ratio': 0.2}
Best RMSE       : 0.3190
Training Time   : 1420.02 seconds


c:\Users\abuba\Downloads\ML-Projects\Day-5\.venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:840: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.583287e+04, tolerance: 2.260e+01
  model = cd_fast.enet_coordinate_descent(


In [55]:
# =============================================================================
# Extract Best Elastic Net Model
# =============================================================================

best_elastic_net = elastic_search.best_estimator_


print("=" * 70)
print("Best Elastic Net Model")
print("=" * 70)

print(best_elastic_net)

print()

print("Best Parameters:")
print(elastic_search.best_params_)

Best Elastic Net Model
ElasticNet(alpha=0.001, l1_ratio=0.2, max_iter=5000, random_state=42)

Best Parameters:
{'alpha': 0.001, 'l1_ratio': 0.2}


In [56]:
# =============================================================================
# Save Tuned Elastic Net Model
# =============================================================================

save_model(
    best_elastic_net,
    "tuned_elastic_net",
    LINEAR_MODELS_DIR,
)

print("Tuned Elastic Net model saved successfully.")


2026-07-23 03:54:12,151 | INFO | Model saved to: c:\Users\abuba\Downloads\ML-Projects\Day-5\models\linear\tuned_elastic_net.joblib


Tuned Elastic Net model saved successfully.


In [57]:
# =============================================================================
# Linear Regression
# =============================================================================

from sklearn.linear_model import LinearRegression

start_time = time.perf_counter()

linear_model = LinearRegression()

linear_model.fit(
    X_train_tune,
    y_train_tune,
)

linear_time = time.perf_counter() - start_time

print("=" * 70)
print("Linear Regression Training Completed")
print("=" * 70)

print(f"Training Time : {linear_time:.2f} seconds")

Linear Regression Training Completed
Training Time : 0.51 seconds


In [58]:
# =============================================================================
# Save Linear Regression Model
# =============================================================================

save_model(
    linear_model,
    "linear_regression",
    LINEAR_MODELS_DIR,
)

print("Linear Regression model saved successfully.")

2026-07-23 03:54:17,339 | INFO | Model saved to: c:\Users\abuba\Downloads\ML-Projects\Day-5\models\linear\linear_regression.joblib


Linear Regression model saved successfully.


In [5]:

from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

LINEAR_MODELS_DIR = PROJECT_ROOT / "models" / "linear"
TREE_MODELS_DIR = PROJECT_ROOT / "models" / "tree"


In [6]:
from pathlib import Path
print(Path.cwd().parent)

c:\Users\abuba\Downloads\ML-Projects\Day-5


In [7]:
# =============================================================================
# Verify Saved Models
# =============================================================================

print("=" * 70)
print("Saved Linear Models")
print("=" * 70)

for model_file in sorted(LINEAR_MODELS_DIR.glob("*.joblib")):
    print(f"✓ {model_file.name}")


print("\n" + "=" * 70)
print("Saved Tree Models")
print("=" * 70)

for model_file in sorted(TREE_MODELS_DIR.glob("*.joblib")):
    print(f"✓ {model_file.name}")

Saved Linear Models
✓ linear_regression.joblib
✓ tuned_elastic_net.joblib
✓ tuned_lasso.joblib
✓ tuned_ridge.joblib

Saved Tree Models
✓ tuned_decision_tree.joblib
✓ tuned_gradient_boosting.joblib
✓ tuned_random_forest.joblib
